# SPAI Phase-2 Training with a CvT-13 Spectral Model — Chest X-raysSecond-phase (supervised) training of **SPAI** — *Any-Resolution AI-Generated Image Detection bySpectral Learning*, CVPR 2025 — on top of a **CvT-13** backbone pre-trained with Masked FrequencyModelling, applied to the medical-imaging domain.The frozen backbone `G` models the spectral distribution of **real** images. This notebook trainsthe three components that sit on top of it:| Component | What it learns ||---|---|| **SRS** — Spectral Reconstruction Similarity | How well `G` reconstructs an image's missing low / high frequencies, as cosine similarity between the features of `x`, `x_low` and `x_high` (mean + std over tokens, `6N` values) || **SCV** — Spectral Context Vector | The spectral context in which those similarities were measured, attended across the `N` blocks || **SCA** — Spectral Context Attention | Fuses the per-patch vectors of an any-resolution image into one image-level vector |Generated images then score as **out-of-distribution** samples of `G`.---## Before you run — Kaggle setup1. **Add Input** → search `nih-chest-xrays/data` → **Add** *(authentic images)*2. **Add Input** → search `whiteflags26/synthetic-chest-x-rays` → **Add** *(generated images)*3. **Settings → Accelerator** → `GPU T4 x2` (or `P100`)4. **Settings → Internet** → **On** *(required for the repo clone, pip and the weights download)*5. Run the cells top to bottom. **Sections 4 and 5 are the gate** — if either fails, stop and   report the traceback before spending GPU hours on the full run.You do **not** need to arrange the Kaggle inputs into any particular folder shape. `/kaggle/input`is read-only, so the `0_real` / `1_fake` tree the training code expects is built for you under`/kaggle/working`.

## 0 · ParametersEverything configurable lives here. Check `REAL_DIR` / `FAKE_DIR` against the tree printed in §1.3.

In [ ]:
# ----------------------------------------------------------------- source code and weightsREPO_URL    = "https://github.com/Kashshaf-Labib/spai-cvt-finetune.git"REPO_BRANCH = "main"WEIGHTS_DRIVE_ID = "180hbXuIVAkXE3iT8BQO3ivoV1mBjz-r7"   # CvT-13 MFM checkpoint (best.pth)# ------------------------------------------------------------------------------- input data# Kaggle mount points do not always match the dataset slug. Section 1.3 lists every directory# under /kaggle/input that holds images, so correct these against what it prints.REAL_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"                  # 112120 -> class 0FAKE_DIR = "/kaggle/input/datasets/whiteflags26/synthetic-chest-x-rays"   #   8550 -> class 1# -------------------------------------------------------------------- acquisition parity# Real and generated images must differ ONLY in content. If they also differ in resolution,# file format or color mode, the detector can separate them on that shortcut alone and the# measured AUC says nothing about generated-image detection. Section 2 audits this; these# settings remove whatever it finds.TARGET_SIZE = 512        # bring both classes to this sizeRESIZE_MODE = "crop"     # "crop": center-crop images that are already large enough, leaving                         #         their spectrum untouched. Prefer this — resampling is a                         #         low-pass filter that attenuates exactly the high frequencies                         #         SPAI depends on, and applying it to only the larger class                         #         plants a resampling cue that separates the classes on its own.                         # "resize": resample everything, so both classes show content at a                         #         comparable scale, at the cost of altering the spectrum.RECODE      = "png"      # re-encode both classes into one format ("png" or "jpeg")TO_GRAY     = True       # discard chrominance (X-rays carry none)# --------------------------------------------------------------------------------- splitsTRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.10, 0.10# NIH filenames are <patient_id>_<followup>.png. Grouping by patient keeps every image of a# patient inside a single split; without it the same patient lands in train AND test, which# inflates the reported score.GROUP_REGEX = r"^(?P<group>[^_]+)_"SEED = 0# ------------------------------------------------------------------------------ pilot runPILOT_MAX_PER_CLASS = 150      # per class -> ~240 train / ~30 val / ~30 test imagesPILOT_EPOCHS        = 2PILOT_WARMUP        = 1PILOT_BATCH         = 4PILOT_ACCUM         = 2# ------------------------------------------------------------------------------- full run# Both classes are sampled down to whichever is smaller, so the ceiling here is the number of# generated images (8550). None = use every image (mind the 20 GB /kaggle/working cap).FULL_MAX_PER_CLASS = 8000FULL_EPOCHS        = 35        # the paper's scheduleFULL_WARMUP        = 5FULL_BATCH         = 8         # 16 GB GPU: each image is 4 patches x 3 spectral branchesFULL_ACCUM         = 4         # -> effective batch 32# The paper uses lr 5e-4 at batch 72. Scaled linearly to an effective batch of 32.FULL_LR            = 2.2e-4FULL_AMP           = "O0"      # "O0" = fp32, safest. Switch to "native" for ~2x speed once                               # the pilot is green. Apex ("O1"/"O2") is not installable here.# ---------------------------------------------------------------------------- runtimeDATA_WORKERS             = 2FEATURE_EXTRACTION_BATCH = 32  # bounds VRAM on the any-resolution val/test pathVAL_BATCH                = 8# ----------------------------------------------------------------------------- layoutWORK      = "/kaggle/working"REPO_DIR  = f"{WORK}/spai-cvt-finetune"WEIGHTS   = f"{REPO_DIR}/weights/best.pth"CFG       = "configs/spai_cvt.yaml"PILOT_DS  = f"{WORK}/datasets/pilot"FULL_DS   = f"{WORK}/datasets/full"PILOT_OUT = f"{WORK}/output/pilot"FULL_OUT  = f"{WORK}/output/full"PILOT_TAG, FULL_TAG = "spai_cvt_pilot", "spai_cvt_full"PILOT_RUN = f"{PILOT_OUT}/finetune/{PILOT_TAG}"FULL_RUN  = f"{FULL_OUT}/finetune/{FULL_TAG}"print("Parameters set. Repository ->", REPO_DIR)

## 1 · Environment### 1.1 Clone the repository and install dependencies

In [ ]:
import os, subprocess, sys, shutilos.environ["SPAI_DISABLE_NEPTUNE"]     = "1"   # no experiment-tracking token on Kaggleos.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"if os.path.isdir(REPO_DIR):    shutil.rmtree(REPO_DIR)subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],               check=True)# albumentations 1.4.x and timm 0.4.12 are the versions the pipeline was written against;# albumentations 2.x removed the `quality_lower` / `var_limit` arguments that it uses.subprocess.run([sys.executable, "-m", "pip", "install", "-q",                "albumentations==1.4.14", "albucore==0.0.16", "timm==0.4.12",                "yacs", "filetype", "gdown", "termcolor", "einops", "torchmetrics"],               check=True)os.chdir(REPO_DIR)sys.path.insert(0, REPO_DIR)print("\nRepository at", os.getcwd())print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

### 1.2 Environment check

In [ ]:
import torch, platformprint(f"python        : {platform.python_version()}")print(f"torch         : {torch.__version__}  (CUDA {torch.version.cuda})")print(f"cuda available: {torch.cuda.is_available()}")if torch.cuda.is_available():    for i in range(torch.cuda.device_count()):        p = torch.cuda.get_device_properties(i)        print(f"  GPU {i}       : {p.name}  {p.total_memory / 1024 ** 3:.1f} GiB")else:    print("  !! No GPU detected. Set Settings -> Accelerator -> GPU before continuing.")for mod in ("timm", "albumentations", "transformers", "torchmetrics"):    try:        print(f"{mod:<14}: {__import__(mod).__version__}")    except Exception as e:        print(f"{mod:<14}: MISSING ({e})")total, _, free = shutil.disk_usage(WORK)print(f"\n/kaggle/working free: {free / 1024 ** 3:.1f} GiB of {total / 1024 ** 3:.1f} GiB")

### 1.3 Locate the attached datasetsConfirm the two paths below match `REAL_DIR` / `FAKE_DIR` in §0. If they differ, edit §0 andre-run it.

In [ ]:
import pathlib, collectionsIMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}def count_images_per_dir(root):    """Maps every directory under `root` to the number of images directly inside it."""    counts = collections.Counter()    for p in pathlib.Path(root).rglob("*"):        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:            counts[p.parent] += 1    return countsdef describe_mounts(base="/kaggle/input", max_rows=25):    """Reports every directory under the Kaggle mounts that actually contains images.    Dataset mount points differ from their slugs often enough that hard-coding them is    unreliable, so the candidates are discovered instead of assumed.    """    base = pathlib.Path(base)    if not base.exists():        print(f"!! {base} does not exist.")        return {}    print(f"Mounted under {base}:")    for p in sorted(base.iterdir()):        print(f"    {p}")    counts = count_images_per_dir(base)    if not counts:        print(f"\n!! No images found anywhere under {base}. Attach the datasets to the "              f"notebook (Add Input) and re-run.")        return {}    # Roll the per-directory counts up into whichever ancestor is the natural dataset root:    # the shallowest directory whose subtree holds images.    totals = collections.Counter()    for directory, n in counts.items():        for ancestor in [directory, *directory.parents]:            if base in ancestor.parents or ancestor == base:                totals[ancestor] += n            if ancestor == base:                break    print(f"\nDirectories containing images (largest first):")    rows = sorted(counts.items(), key=lambda kv: (-kv[1], str(kv[0])))[:max_rows]    for directory, n in rows:        print(f"    {n:>8} images   {directory}")    if len(counts) > max_rows:        print(f"    ... and {len(counts) - max_rows} more directories")    print(f"\nRolled-up totals per candidate root:")    candidates = {d: n for d, n in totals.items()                  if len(d.relative_to(base).parts) <= 2 and n > 0}    for directory, n in sorted(candidates.items(), key=lambda kv: (-kv[1], str(kv[0]))):        print(f"    {n:>8} images   {directory}")    return candidatesdef preview(root):    """Prints the image counts of one chosen root."""    root = pathlib.Path(root)    if not root.exists():        print(f"  !! {root} does not exist")        return 0    counts = count_images_per_dir(root)    total = sum(counts.values())    for directory, n in sorted(counts.items(), key=lambda kv: (-kv[1], str(kv[0])))[:8]:        rel = directory.relative_to(root) if directory != root else pathlib.Path(".")        print(f"    {str(rel):<52} {n:>8} images")    if len(counts) > 8:        print(f"    ... and {len(counts) - 8} more directories")    print(f"    {'TOTAL':<52} {total:>8} images")    return totalcandidates = describe_mounts()print(f"\nREAL_DIR = {REAL_DIR}")n_real = preview(REAL_DIR)print(f"\nFAKE_DIR = {FAKE_DIR}")n_fake = preview(FAKE_DIR)if n_real == 0 or n_fake == 0:    # Deliberately a RuntimeError rather than SystemExit: IPython swallows SystemExit, so a    # "Run All" would carry on into the following sections with unusable paths.    raise RuntimeError(        "REAL_DIR and/or FAKE_DIR hold no images. Copy the two correct paths from the "        "'Rolled-up totals per candidate root' list above into section 0, RE-RUN SECTION 0 so "        "that the new values take effect, then re-run this cell. The two roots must be "        "different and neither may contain the other."    )real_path = pathlib.Path(REAL_DIR).resolve()fake_path = pathlib.Path(FAKE_DIR).resolve()assert real_path != fake_path, "REAL_DIR and FAKE_DIR point at the same directory."# Nesting one root inside the other would silently pull the generated images into the# authentic class, and the labels would be wrong without anything failing.assert fake_path not in real_path.parents and real_path not in fake_path.parents, \    (f"One root contains the other ({real_path} / {fake_path}), so one class would include "     f"the images of the other. Point them at two sibling directories.")print(f"\nOK - {n_real} authentic and {n_fake} generated images.")

### 1.4 Download the CvT-13 spectral modelThe checkpoint produced by the masked-frequency-modelling pre-training stage.

In [ ]:
import gdownpathlib.Path(f"{REPO_DIR}/weights").mkdir(parents=True, exist_ok=True)if not pathlib.Path(WEIGHTS).exists():    gdown.download(id=WEIGHTS_DRIVE_ID, output=WEIGHTS, quiet=False)size_mb = pathlib.Path(WEIGHTS).stat().st_size / 1024 ** 2print(f"\ncheckpoint: {WEIGHTS}  ({size_mb:.1f} MB)")assert size_mb > 200, ("The download is far smaller than expected (~228 MB). Google Drive most "                       "likely returned an HTML quota page instead of the file.")ckpt = torch.load(WEIGHTS, map_location="cpu", weights_only=False)encoder_keys = [k for k in ckpt["model"] if k.startswith("encoder.encoder.")]print(f"pre-training epoch  : {ckpt.get('epoch')}")print(f"best recon. loss    : {ckpt.get('best_loss'):.4f}")print(f"encoder tensors     : {len(encoder_keys)}  (expected 455)")print(f"masking radius (r)  : {ckpt['config'].DATA.MASK_RADIUS1}  "      f"(must be 16 so that phase 2 aligns with the pretext task)")assert len(encoder_keys) == 455, "Unexpected checkpoint layout."del ckpt

## 2 · Data audit — is there a shortcut in the data?**This is the most important methodological check in the notebook.**SPAI is meant to detect the *spectral* inconsistencies a generative model introduces. If theauthentic and generated images also differ in resolution, file format or colour mode, a classifiercan reach a high AUC by detecting **that** instead, and the number would say nothing aboutgenerated-image detection.The cell below reports the acquisition properties of both classes and warns on any systematicdifference. Section 3 then removes whatever it finds.**If the audit reports a resolution mismatch**, note how §3 equalises it. `RESIZE_MODE="crop"`(the default) center-crops the larger class rather than resampling it, because resampling is alow-pass filter acting on exactly the high frequencies this method reads — applying it to oneclass only would replace one shortcut with another. The trade-off is that the two classes thendepict content at different scales. `RESIZE_MODE="resize"` makes the opposite trade. Reportwhichever you used.

In [ ]:
def run(cmd, cwd=REPO_DIR, check=True):    """Runs a command, streaming its output into the notebook."""    print("$ " + " ".join(str(c) for c in cmd) + "\n", flush=True)    env = {**os.environ, "PYTHONPATH": REPO_DIR}    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=env, text=True,                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)    for line in proc.stdout:        print(line, end="")    proc.wait()    if check and proc.returncode != 0:        raise RuntimeError(f"Command failed with exit code {proc.returncode}")    return proc.returncodedef require_inputs():    """Fails early, and legibly, when the dataset roots are not usable."""    for name, path in (("REAL_DIR", REAL_DIR), ("FAKE_DIR", FAKE_DIR)):        if not pathlib.Path(path).is_dir():            raise RuntimeError(                f"{name} = {path!r} is not a directory. Correct it in section 0, RE-RUN "                f"SECTION 0 so the new value takes effect, then re-run section 1.3."            )require_inputs()print(f"REAL_DIR = {REAL_DIR}\nFAKE_DIR = {FAKE_DIR}\n")run([sys.executable, "-m", "spai.tools.prepare_medical_dataset",     "--real-dir", REAL_DIR, "--fake-dir", FAKE_DIR, "-o", "/tmp/unused",     "--audit-sample", 800, "--audit-only"])

## 3 · Build the pilot datasetArranges a small balanced subset into the layout the training code expects, applying the paritytransformations chosen in §0:```datasets/pilot/├── train/{0_real, 1_fake}/├── val/{0_real, 1_fake}/└── test/{0_real, 1_fake}/```then writes the `image,class,split` CSVs with the repository's own `create_dir_csv` tool.

In [ ]:
import pandas as pddef build_dataset(out_dir, max_per_class, tag):    """Arranges both classes into the 0_real/1_fake tree and writes the dataset CSVs."""    if os.path.isdir(out_dir):        shutil.rmtree(out_dir)    cmd = [sys.executable, "-m", "spai.tools.prepare_medical_dataset",           "--real-dir", REAL_DIR, "--fake-dir", FAKE_DIR, "-o", out_dir,           "--train-ratio", TRAIN_RATIO, "--val-ratio", VAL_RATIO, "--test-ratio", TEST_RATIO,           "--group-regex", GROUP_REGEX, "--seed", SEED, "--audit-sample", 200]    if max_per_class is not None:        cmd += ["--max-per-class", max_per_class]    if TARGET_SIZE is not None:        cmd += ["--target-size", TARGET_SIZE, "--resize-mode", RESIZE_MODE]    if RECODE is not None:        cmd += ["--recode", RECODE]    if TO_GRAY:        cmd += ["--to-gray"]    run(cmd)    # One CSV holds both the train and the val split, while the test split is kept separate so    # that it is only ever touched at evaluation time.    run([sys.executable, "-m", "spai.tools.create_dir_csv",         "--train_dir", f"{out_dir}/train", "--val_dir", f"{out_dir}/val",         "-o", f"{out_dir}/train_val.csv", "-r", out_dir])    run([sys.executable, "-m", "spai.tools.create_dir_csv",         "--test_dir", f"{out_dir}/test", "-o", f"{out_dir}/test.csv", "-r", out_dir])    for name in ("train_val", "test"):        df = pd.read_csv(f"{out_dir}/{name}.csv")        print(f"\n{tag}/{name}.csv")        print(df.groupby(["split", "class"]).size().rename("images").to_frame())    verify_parity(out_dir)def verify_parity(out_dir):    """Re-audits the arranged images, to confirm the two classes now share their properties.    The audit of section 2 describes the source images; this one describes what the model will    actually be trained on.    """    from spai.tools.prepare_medical_dataset import audit_images, find_images, report_parity    print("\n=== Acquisition properties AFTER preparation (train split) ===")    audits = {}    for cls, label in (("0_real", "authentic (class 0)"), ("1_fake", "generated (class 1)")):        audits[cls] = audit_images(find_images(pathlib.Path(f"{out_dir}/train/{cls}")), 400)        a = audits[cls]        print(f"  {label}")        for field in ("sizes", "formats", "modes"):            rendered = ", ".join(                (f"{k[0]}x{k[1]}" if isinstance(k, tuple) else str(k)) + f": {v}"                for k, v in a[field].most_common(4))            print(f"    {field:<9}: {rendered}")    remaining = report_parity(audits["0_real"], audits["1_fake"])    if remaining:        for w in remaining:            print(f"\n  [WARNING] {w}")        print("\n  >>> The classes still differ. Revisit TARGET_SIZE / RECODE / TO_GRAY in "              "section 0 before training, or the result will not be interpretable.")    else:        print("\n  OK - the two classes now share their resolution, format and colour mode.")build_dataset(PILOT_DS, PILOT_MAX_PER_CLASS, "pilot")

## 4 · Smoke testBuilds the architecture, loads the CvT weights and pushes a tensor of each shape the training loopproduces through it. Takes well under a minute and catches integration problems before any GPU timeis spent.

In [ ]:
import loggingfrom torch import nnfrom spai.config import get_custom_configfrom spai.models import build_cls_modelfrom spai.utils import load_pretrainedlogging.basicConfig(level=logging.INFO, format="%(message)s")log = logging.getLogger("smoke")cfg = get_custom_config(CFG)cfg.defrost(); cfg.PRETRAINED = WEIGHTS; cfg.freeze()model = build_cls_model(cfg)backbone = model.get_vision_transformer()print(f"\nmodel    : {type(model).__name__}")print(f"backbone : {type(backbone).__name__}  "      f"blocks={backbone.num_features}  dim={backbone.embed_dim}")load_pretrained(cfg, backbone, log)n_all = sum(p.numel() for p in model.parameters())n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"\nparameters: {n_all / 1e6:.2f}M total | {n_tr / 1e6:.2f}M trainable "      f"| {(n_all - n_tr) / 1e6:.2f}M frozen (the spectral model)")# The backbone must stay in evaluation mode even while the model trains. A CvT normalises its# convolutional projections with batch normalisation, which would otherwise normalise by batch# statistics and mutate its running statistics, altering the frozen spectral model every step.model.train()bns = [m for m in backbone.modules() if isinstance(m, nn.BatchNorm2d)]assert not any(m.training for m in bns), "The frozen backbone was left in training mode."print(f"batch-norm: {len(bns)} layers, all held in eval mode while model.train() -> OK")device = "cuda" if torch.cuda.is_available() else "cpu"model = model.to(device)# Training shape: DATA.AUGMENTED_VIEWS views concatenated horizontally.out = model(torch.rand(2, 3, 224, 224 * cfg.DATA.AUGMENTED_VIEWS, device=device))loss = nn.BCEWithLogitsLoss()(out.squeeze(1), torch.tensor([0., 1.], device=device))loss.backward()gnorm = sum(p.grad.norm() ** 2 for p in model.parameters() if p.grad is not None).sqrt()print(f"train pass: logits {tuple(out.shape)} | loss {loss.item():.4f} | grad-norm {gnorm:.2f}")# Inference shape: a list of images of arbitrary and differing resolutions.model.eval()with torch.no_grad():    out = model([torch.rand(1, 3, 512, 512, device=device),                 torch.rand(1, 3, 448, 672, device=device)], FEATURE_EXTRACTION_BATCH)print(f"infer pass: logits {tuple(out.shape)} | scores "      f"{[round(s, 4) for s in torch.sigmoid(out).squeeze(1).tolist()]}")del model, backbonetorch.cuda.empty_cache()print("\nSMOKE TEST PASSED")

## 5 · Pilot runTwo epochs on a few hundred images. **This is the gate** — if it completes and the loss moves, thefull run will work. If it raises, stop here and report the traceback.

In [ ]:
def train_cmd(dataset_dir, output_dir, tag, epochs, warmup, batch, accum, lr, amp,              resume=None, max_kept_checkpoints=2):    cmd = [sys.executable, "-m", "spai", "train",           "--cfg", CFG,           "--batch-size", batch,           "--accumulation-steps", accum,           "--learning-rate", lr,           "--pretrained", WEIGHTS,           "--output", output_dir,           "--data-path", f"{dataset_dir}/train_val.csv",           "--csv-root-dir", dataset_dir,           "--tag", tag,           "--amp-opt-level", amp,           "--data-workers", DATA_WORKERS,           "--max-kept-checkpoints", max_kept_checkpoints,           "--opt", "TRAIN.EPOCHS", str(epochs),           "--opt", "TRAIN.WARMUP_EPOCHS", str(warmup),           "--opt", "DATA.VAL_BATCH_SIZE", str(VAL_BATCH),           "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH)]    if resume:        cmd += ["--resume", resume]    return cmd# The pilot always starts fresh, so that re-running this cell actually re-runs it rather than# finding a completed run and returning immediately.shutil.rmtree(PILOT_OUT, ignore_errors=True)run(train_cmd(PILOT_DS, PILOT_OUT, PILOT_TAG,              PILOT_EPOCHS, PILOT_WARMUP, PILOT_BATCH, PILOT_ACCUM, 2.2e-4, "O0"))

### 5.1 Pilot learning curves

In [ ]:
import matplotlib.pyplot as pltimport matplotlib as mplfrom matplotlib.ticker import MaxNLocator# Chart palette: validated categorical slots (blue, orange, aqua) on a light surface.SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e2e1dd"SERIES = {"a": "#2a78d6", "b": "#eb6834", "c": "#1baf7a"}mpl.rcParams.update({    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,    "axes.edgecolor": GRID, "axes.labelcolor": INK_MUTED,    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,    "axes.spines.top": False, "axes.spines.right": False,    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,})def place_end_labels(ax, labels, min_gap_frac=0.055):    """Draws a label at the right end of each series, nudged apart so none overlap.    `labels` is a sequence of (y, text) pairs. Series whose final values coincide would    otherwise print on top of each other and become unreadable.    """    lo, hi = ax.get_ylim()    min_gap = (hi - lo) * min_gap_frac    ordered = sorted(labels, key=lambda t: t[0])    placed = []    for y, text in ordered:        if placed and y - placed[-1][0] < min_gap:            y = placed[-1][0] + min_gap        placed.append([y, text])    # Nudging upwards can push the topmost label past the axis limit, where it would be    # clipped away. The whole cluster is shifted back down by however much it overflows.    overflow = placed[-1][0] - (hi - min_gap * 0.5)    if overflow > 0:        shift = min(overflow, max(placed[0][0] - (lo + min_gap * 0.5), 0))        for item in placed:            item[0] -= shift    x_end = ax.get_xlim()[1]    for y, text in placed:        ax.annotate(text, (x_end, y), fontsize=9, color=INK_MUTED, va="center", ha="right",                    annotation_clip=False)def plot_history(run_dir, title):    """Loss curves and validation metrics on separate axes sharing one epoch scale.    Loss and the ranking metrics are deliberately not drawn against two y-scales of a single    plot: a dual-axis chart invites reading a crossing point that carries no meaning.    """    hist = pd.read_csv(f"{run_dir}/history.csv")    fig, (ax_loss, ax_met) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)    # Headroom on the right for the annotations, and whole-number epoch ticks.    first, last = hist.epoch.iloc[0], hist.epoch.iloc[-1]    pad = max((last - first) * 0.28, 0.6)    for ax in (ax_loss, ax_met):        ax.set_xlim(first - max((last - first) * 0.02, 0.05), last + pad)        ax.xaxis.set_major_locator(MaxNLocator(integer=True))        # The padding exists only to fit the annotations, so no tick is drawn inside it.        ax.set_xticks([t for t in ax.get_xticks() if first <= t <= last])    ax_loss.plot(hist.epoch, hist.train_loss, color=SERIES["a"], lw=2, label="train")    ax_loss.plot(hist.epoch, hist.val_loss, color=SERIES["b"], lw=2, label="validation")    best = int(hist.val_loss.idxmin())    ax_loss.scatter([hist.epoch[best]], [hist.val_loss[best]], s=42, zorder=5,                    color=SERIES["b"], edgecolor=SURFACE, linewidth=2)    ax_loss.annotate(f"best epoch {int(hist.epoch[best])}\n{hist.val_loss[best]:.4f}",                     (hist.epoch[best], hist.val_loss[best]), textcoords="offset points",                     xytext=(8, 8), fontsize=9, color=INK_MUTED,                     annotation_clip=False)    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("BCE loss")    ax_loss.set_title("Loss", loc="left", color=INK)    ax_loss.legend(loc="upper right")    end_labels = []    for key, label, color in (("val_auc", "AUC", SERIES["a"]),                              ("val_ap", "AP", SERIES["b"]),                              ("val_accuracy", "accuracy", SERIES["c"])):        ax_met.plot(hist.epoch, hist[key], color=color, lw=2, label=label)        end_labels.append((hist[key].iloc[-1], f"{label} {hist[key].iloc[-1]:.3f}"))    ax_met.set_xlabel("epoch"); ax_met.set_ylabel("score"); ax_met.set_ylim(0, 1.02)    ax_met.set_title("Validation metrics", loc="left", color=INK)    ax_met.legend(loc="lower right", ncol=3)    # Direct end labels, so identity never rests on colour alone.    place_end_labels(ax_met, end_labels)    fig.suptitle(title, x=0.006, ha="left", fontsize=13, color=INK)    plt.show()    print(f"final train loss {hist.train_loss.iloc[-1]:.4f} | "          f"final val loss {hist.val_loss.iloc[-1]:.4f} | "          f"gap {hist.val_loss.iloc[-1] - hist.train_loss.iloc[-1]:+.4f}")    print(f"best val loss {hist.val_loss.min():.4f} at epoch {int(hist.epoch[best])} | "          f"best val AUC {hist.val_auc.max():.4f}")    return histpilot_history = plot_history(PILOT_RUN, "Pilot run — 2 epochs")# Project the duration of the full run, so that a schedule that cannot fit inside a Kaggle# session is caught before it is launched rather than at hour 12.pilot_train = len(pd.read_csv(f"{PILOT_DS}/train_val.csv").query("split == 'train'"))full_train = int(2 * (FULL_MAX_PER_CLASS or 8550) * TRAIN_RATIO)per_epoch = pilot_history.epoch_time.iloc[-1] * full_train / pilot_trainprint(f"\npilot: {pilot_train} train images at {pilot_history.epoch_time.iloc[-1]:.0f}s/epoch")print(f"full : {full_train} train images -> ~{per_epoch / 60:.1f} min/epoch, "      f"~{per_epoch * FULL_EPOCHS / 3600:.1f} h for {FULL_EPOCHS} epochs")if per_epoch * FULL_EPOCHS / 3600 > 10:    print("\n  [WARNING] That projection exceeds a comfortable margin under Kaggle's 12 h "          "session limit.\n  Lower FULL_EPOCHS or FULL_MAX_PER_CLASS, or set FULL_AMP = "          "'native' for roughly a 2x speed-up.\n  The full run is restartable either way, so "          "an overrun is recoverable rather than fatal.")

> **Stop here if the pilot failed.** Report the traceback and it can be fixed before you spend GPU> hours. If it completed, the numbers do not need to be good yet — two epochs on a few hundred> images only proves that the pipeline runs.---## 6 · Full run### 6.1 Build the full dataset

In [ ]:
# Rough disk estimate before committing: a 512x512 grayscale PNG is ~0.15-0.25 MB.if FULL_MAX_PER_CLASS is not None:    est_gb = 2 * FULL_MAX_PER_CLASS * 0.22 / 1024    free_gb = shutil.disk_usage(WORK).free / 1024 ** 3    print(f"estimated dataset size: {est_gb:.1f} GiB | free: {free_gb:.1f} GiB")    assert est_gb < free_gb * 0.7, ("Not enough free space. Lower FULL_MAX_PER_CLASS in "                                    "section 0.")if os.path.isdir(PILOT_DS):    shutil.rmtree(PILOT_DS)          # the pilot images are no longer neededbuild_dataset(FULL_DS, FULL_MAX_PER_CLASS, "full")

### 6.2 TrainThe paper's schedule: 35 epochs, 5 warm-up epochs, cosine decay, AdamW, BCE.**This cell is restartable.** It passes `--resume auto`, and the trainer rewrites three thingsafter every epoch:| File | Purpose ||---|---|| `latest.pth` | the resume point — model, optimizer, scheduler, epoch, AMP scale || `best.pth` | the epoch with the lowest validation loss, i.e. the one §7 evaluates || `history.csv` | the learning curves, including the epochs completed before an interruption |So if the kernel dies, you hit an OOM, or you simply interrupt the cell, **just run it again** —it continues from the last completed epoch instead of starting over, and the curves staycontinuous. A run that finds no checkpoint starts from scratch, so the same command works eitherway.One limit worth knowing: `/kaggle/working` survives a kernel restart *within* a session, but notthe end of the session itself. To carry a run across sessions, use **Save Version** so thecheckpoints become notebook output you can attach back as an input.Kaggle caps sessions at 12 h. After the pilot, check the projected time with the arithmetic in§5.1 before launching this.

In [ ]:
run(train_cmd(FULL_DS, FULL_OUT, FULL_TAG,              FULL_EPOCHS, FULL_WARMUP, FULL_BATCH, FULL_ACCUM, FULL_LR, FULL_AMP,              resume="auto"))

### 6.3 Learning curves — the overfitting readoutRead the **gap** between the two loss curves. Train loss falling while validation loss flattens andthen rises is overfitting; the epoch marked *best* is the one the paper's protocol selects.

In [ ]:
history = plot_history(FULL_RUN, f"Full run — {FULL_EPOCHS} epochs")

## 7 · Evaluation on the held-out test splitThe test split was seen neither during training nor during model selection, and patients in itappear in no other split.

In [ ]:
import glob, re# The trainer writes the epoch with the lowest validation loss under a stable name.BEST_CKPT = f"{FULL_RUN}/best.pth"if not os.path.exists(BEST_CKPT):    # Fall back to the epoch-named checkpoints, which a run started before best.pth existed    # would have produced. Only improving epochs are saved, so the last one is the best one.    ckpts = sorted(glob.glob(f"{FULL_RUN}/ckpt_epoch_*.pth"),                   key=lambda p: int(re.search(r"ckpt_epoch_(\d+)", p).group(1)))    assert ckpts, f"No checkpoint found under {FULL_RUN}"    BEST_CKPT = ckpts[-1]best_epoch = int(torch.load(BEST_CKPT, map_location="cpu", weights_only=False)["epoch"])# Read the history from disk rather than relying on section 6.3 having been run in this kernel.run_history = pd.read_csv(f"{FULL_RUN}/history.csv")selected = run_history.loc[run_history.epoch == best_epoch]if len(selected):    row = selected.iloc[0]    print(f"evaluating {os.path.basename(BEST_CKPT)} - epoch {best_epoch}, "          f"val loss {row.val_loss:.4f}, val AUC {row.val_auc:.4f}\n")else:    print(f"evaluating {os.path.basename(BEST_CKPT)} - epoch {best_epoch}\n")TEST_OUT = f"{WORK}/output/test"run([sys.executable, "-m", "spai", "test",     "--cfg", CFG,     "--batch-size", VAL_BATCH,     "--model", BEST_CKPT,     "--output", TEST_OUT,     "--tag", FULL_TAG,     "--test-csv", f"{FULL_DS}/test.csv",     "--test-csv-root-dir", FULL_DS,     "--update-csv",                      # writes per-image scores, used by the next cell     "--opt", "DATA.NUM_WORKERS", str(DATA_WORKERS),     "--opt", "MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH)])

### 7.1 Score distributions and ROCThe left panel is the direct picture of the paper's claim: generated images should sit as**out-of-distribution** samples of the spectral model and separate from the authentic ones.

In [ ]:
import numpy as npfrom sklearn.metrics import roc_curve, roc_auc_score, average_precision_score, accuracy_scorescored = pd.read_csv(f"{TEST_OUT}/finetune/{FULL_TAG}/test.csv")score_col = [c for c in scored.columns if c.startswith(FULL_TAG)][0]scored = scored[pd.to_numeric(scored[score_col], errors="coerce").notna()]y = scored["class"].astype(int).to_numpy()s = scored[score_col].astype(float).to_numpy()auc = roc_auc_score(y, s)ap = average_precision_score(y, s)acc = accuracy_score(y, (s >= 0.5).astype(int))fig, (ax_hist, ax_roc) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)bins = np.linspace(0, 1, 41)ax_hist.hist(s[y == 0], bins=bins, color=SERIES["a"], alpha=0.75, label="authentic")ax_hist.hist(s[y == 1], bins=bins, color=SERIES["b"], alpha=0.75, label="generated")ax_hist.axvline(0.5, color=INK_MUTED, lw=1, ls="--")ax_hist.set_xlabel("predicted score"); ax_hist.set_ylabel("images")ax_hist.set_title("Score distribution", loc="left", color=INK)ax_hist.legend(loc="upper center")fpr, tpr, _ = roc_curve(y, s)ax_roc.plot([0, 1], [0, 1], color=GRID, lw=1.5, ls="--")ax_roc.plot(fpr, tpr, color=SERIES["a"], lw=2)ax_roc.annotate(f"AUC {auc:.3f}", (0.55, 0.18), fontsize=11, color=INK)ax_roc.set_xlabel("false positive rate"); ax_roc.set_ylabel("true positive rate")ax_roc.set_title("ROC", loc="left", color=INK)ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.02)fig.suptitle("Held-out test split", x=0.006, ha="left", fontsize=13, color=INK)plt.show()print(f"images   : {len(y)}  ({int((y == 0).sum())} authentic / {int((y == 1).sum())} generated)")print(f"AUC      : {auc:.4f}")print(f"AP       : {ap:.4f}")print(f"Accuracy : {acc:.4f}  (threshold 0.5)")print(f"\nmedian score | authentic {np.median(s[y == 0]):.4f} "      f"| generated {np.median(s[y == 1]):.4f}")

## 8 · Save artifacts

In [ ]:
ART = f"{WORK}/artifacts"os.makedirs(ART, exist_ok=True)shutil.copy(BEST_CKPT, f"{ART}/spai_cvt_best.pth")for src, dst in ((f"{FULL_RUN}/history.csv", "history.csv"),                 (f"{FULL_RUN}/config.json", "config.json"),                 (f"{TEST_OUT}/finetune/{FULL_TAG}/test.csv", "test_scores.csv")):    if os.path.exists(src):        shutil.copy(src, f"{ART}/{dst}")with open(f"{ART}/summary.txt", "w") as f:    f.write("SPAI phase-2 with a CvT-13 spectral model\n"            f"checkpoint : {os.path.basename(BEST_CKPT)}\n"            f"images     : {FULL_MAX_PER_CLASS} per class (balanced)\n"            f"parity     : size={TARGET_SIZE} ({RESIZE_MODE}) recode={RECODE} "            f"gray={TO_GRAY}\n"            f"schedule   : {FULL_EPOCHS} epochs | batch {FULL_BATCH} x accum {FULL_ACCUM} "            f"| lr {FULL_LR} | amp {FULL_AMP}\n"            f"test AUC   : {auc:.4f}\n"            f"test AP    : {ap:.4f}\n"            f"test Acc   : {acc:.4f}\n")# The arranged dataset is large and fully reproducible from this notebook, so it is dropped to# keep the session output within Kaggle's size limit.shutil.rmtree(FULL_DS, ignore_errors=True)print("Saved to", ART)for p in sorted(pathlib.Path(ART).iterdir()):    print(f"  {p.name:<24} {p.stat().st_size / 1024 ** 2:8.2f} MB")print()print(open(f"{ART}/summary.txt").read())

---## Reading the result- **Domain shift.** The spectral model `G` was pre-trained on natural images (an ImageNet subset)  but is used here to model the spectral distribution of chest X-rays. The paper's premise is that  `G` captures the spectral distribution of *real* images; a natural-image `G` may model X-ray  spectra only approximately. If the AUC disappoints, the highest-value follow-up is repeating the  phase-1 MFM pre-training on real X-rays.- **Grayscale.** X-rays carry no chrominance, so the colour-related spectral cues available in the  paper's setting are absent here. Expect a lower absolute AUC than the reported 91.0.- **A single generator.** With one synthetic source this measures in-domain detection only. It does  **not** test SPAI's central claim of generalising to unseen generators — that would need a second  generator held out entirely from training.- **Parity.** Whatever §2 reported, state it in the write-up alongside the transformations applied  in §3. The first question a reader asks about any AI-generated-image detector is whether the two  classes differ in some trivial way.